In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import zipfile
import io
import requests
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# --- 1. Load Data (Robust Method) ---

file_path = 'SMSSpamCollection'
zip_name = 'smsspamcollection.zip'
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

if os.path.exists(file_path):
    print(f"Found dataset locally at {file_path}. Loading...")
    df = pd.read_csv(file_path, sep="\t", header=None, names=["label", "message"])
else:
    print(f"Dataset not found locally. Downloading from {url}...")
    try:
        response = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(response.content))
        z.extractall(".")
        print("Download and extraction complete.")
        df = pd.read_csv(file_path, sep="\t", header=None, names=["label", "message"])
    except Exception as e:
        print(f"Error downloading data: {e}")
        raise e

print("Data loaded successfully.")
print(f"Shape: {df.shape}")
display(df.head())

Dataset not found locally. Downloading from https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip...
Download and extraction complete.
Data loaded successfully.
Shape: (5572, 2)


,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [3]:
# --- 2. Preprocessing ---

# Features and target
X = df["message"].astype(str)
y = df["label"].map({"ham": 0, "spam": 1})

# Train-test split
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Training samples: {len(X_tr)}")
print(f"Testing samples: {len(X_te)}")

Training samples: 4179
Testing samples: 1393


In [4]:
# --- 3. Model: Naive Bayes ---

print("Training Naive Bayes...")
nb_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", MultinomialNB())
])

nb_model.fit(X_tr, y_tr)
y_pred_nb = nb_model.predict(X_te)

print("=== Naive Bayes (Spam Classification) ===")
print("Accuracy:", accuracy_score(y_te, y_pred_nb))
print("\nClassification Report:\n",
      classification_report(y_te, y_pred_nb, target_names=["Non-Spam", "Spam"]))
print("Confusion Matrix:\n", confusion_matrix(y_te, y_pred_nb))

Training Naive Bayes...
=== Naive Bayes (Spam Classification) ===
Accuracy: 0.9705671213208902

Classification Report:
               precision    recall  f1-score   support

    Non-Spam       0.97      1.00      0.98      1206
        Spam       1.00      0.78      0.88       187

    accuracy                           0.97      1393
   macro avg       0.98      0.89      0.93      1393
weighted avg       0.97      0.97      0.97      1393

Confusion Matrix:
 [[1206    0]
 [  41  146]]


In [5]:
# --- 4. Model: Decision Tree ---

print("Training Decision Tree...")
dt_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", DecisionTreeClassifier(
        criterion="entropy",
        max_depth=25,
        min_samples_leaf=5,
        random_state=42
    ))
])

dt_model.fit(X_tr, y_tr)
dt_pred = dt_model.predict(X_te)

print("=== Decision Tree (Entropy) — Spam Classification ===")
print("Accuracy:", accuracy_score(y_te, dt_pred))
print("\nClassification Report:\n", classification_report(
    y_te, dt_pred, target_names=["Non-Spam", "Spam"], zero_division=0
))
print("Confusion Matrix:\n", confusion_matrix(y_te, dt_pred))

Training Decision Tree...
=== Decision Tree (Entropy) — Spam Classification ===
Accuracy: 0.9425699928212491

Classification Report:
               precision    recall  f1-score   support

    Non-Spam       0.96      0.98      0.97      1206
        Spam       0.82      0.73      0.77       187

    accuracy                           0.94      1393
   macro avg       0.89      0.85      0.87      1393
weighted avg       0.94      0.94      0.94      1393

Confusion Matrix:
 [[1177   29]
 [  51  136]]


In [6]:
# --- 5. Model: ANN (MLP) ---

print("Training MLP Classifier...")
ann_model = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("svd", TruncatedSVD(n_components=200, random_state=42)),
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        max_iter=50,  # Increased slightly for better convergence
        random_state=42
    ))
])

ann_model.fit(X_tr, y_tr)
ann_pred = ann_model.predict(X_te)

print("=== ANN (MLP) — Spam Classification ===")
print("Accuracy:", accuracy_score(y_te, ann_pred))
print("\nClassification Report:\n", classification_report(
    y_te, ann_pred, target_names=["Non-Spam", "Spam"], zero_division=0
))
print("Confusion Matrix:\n", confusion_matrix(y_te, ann_pred))

Training MLP Classifier...
=== ANN (MLP) — Spam Classification ===
Accuracy: 0.9813352476669059

Classification Report:
               precision    recall  f1-score   support

    Non-Spam       0.99      0.99      0.99      1206
        Spam       0.95      0.90      0.93       187

    accuracy                           0.98      1393
   macro avg       0.97      0.95      0.96      1393
weighted avg       0.98      0.98      0.98      1393

Confusion Matrix:
 [[1198    8]
 [  18  169]]


In [7]:
# --- 6. Sample Prediction ---

sample_msg = "Congratulations! You have won a free prize. Call now!"
print("\nSample Message:", sample_msg)

pred_nb_sample = nb_model.predict([sample_msg])[0]
print("Naive Bayes Prediction:", "Spam" if pred_nb_sample == 1 else "Non-Spam")

pred_dt_sample = dt_model.predict([sample_msg])[0]
print("Decision Tree Prediction:", "Spam" if pred_dt_sample == 1 else "Non-Spam")

pred_ann_sample = ann_model.predict([sample_msg])[0]
print("MLP Prediction:", "Spam" if pred_ann_sample == 1 else "Non-Spam")


Sample Message: Congratulations! You have won a free prize. Call now!
Naive Bayes Prediction: Spam
Decision Tree Prediction: Non-Spam
MLP Prediction: Spam
